# SQL Interview Preparation — Runnable Examples

This notebook uses SQLite and small synthetic tables, so no external data download or database server is required.

**How to use it:** run cells in order, inspect results, then modify the queries to test edge cases such as NULLs, duplicate values, and ties.

In [ ]:
import sqlite3
import pandas as pd

# In-memory SQLite database: it resets when the notebook kernel restarts.
conn = sqlite3.connect(":memory:")

def show(sql):
    """Execute a read-only query and display the returned rows."""
    return pd.read_sql_query(sql, conn)


## 1. Create sample tables

In [ ]:
conn.executescript('''
PRAGMA foreign_keys = ON;

CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    manager_id INTEGER,
    salary REAL,
    hire_date TEXT,
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (manager_id) REFERENCES employees(employee_id)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    category TEXT NOT NULL
);
''')

conn.executemany(
    "INSERT INTO departments VALUES (?, ?)",
    [(10, "Analytics"), (20, "Engineering"), (30, "Sales"), (40, "Legal")]
)

conn.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?)",
    [
        (1, "Asha", 10, None, 95000, "2021-02-15"),
        (2, "Ravi", 20, 1, 88000, "2022-06-01"),
        (3, "Mina", 10, 1, 95000, "2023-01-10"),
        (4, "Omar", 30, 1, 72000, "2020-09-21"),
        (5, "Leah", 20, 2, 68000, "2024-03-05"),
        (6, "Arun", None, 1, 81000, "2023-11-11"),
        (7, "Isha", 30, 4, None, "2024-07-19"),
    ]
)

conn.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
    [
        (1001, 501, "2025-01-01", 120.0, "Books"),
        (1002, 502, "2025-01-01", 75.0, "Home"),
        (1003, 501, "2025-01-03", 45.0, "Books"),
        (1004, 503, "2025-01-04", 210.0, "Electronics"),
        (1005, 502, "2025-01-04", 30.0, "Home"),
        (1006, 501, "2025-01-05", 60.0, "Electronics"),
    ]
)

print("Tables created and sample rows inserted.")


## 2. SELECT, filtering, sorting, and NULLs

In [ ]:
show('''
SELECT employee_id, name, salary
FROM employees
WHERE salary >= 80000
ORDER BY salary DESC, name;
''')

In [ ]:
# NULL must be checked with IS NULL, not = NULL.
show("SELECT employee_id, name FROM employees WHERE department_id IS NULL;")

## 3. Aggregation: GROUP BY and HAVING

In [ ]:
show('''
SELECT department_id,
       COUNT(*) AS employee_count,
       COUNT(salary) AS employees_with_salary,
       AVG(salary) AS average_salary
FROM employees
GROUP BY department_id
HAVING COUNT(*) >= 1
ORDER BY average_salary DESC;
''')

`COUNT(*)` counts rows, while `COUNT(salary)` ignores NULL salaries. `HAVING` filters groups after aggregation.

## 4. Joins

In [ ]:
# Preserve employees even when there is no matching department.
show('''
SELECT e.name, d.department_name
FROM employees AS e
LEFT JOIN departments AS d
  ON e.department_id = d.department_id
ORDER BY e.employee_id;
''')

In [ ]:
# Self join: employee and manager names.
show('''
SELECT e.name AS employee_name, m.name AS manager_name
FROM employees AS e
LEFT JOIN employees AS m
  ON e.manager_id = m.employee_id
ORDER BY e.employee_id;
''')

## 5. CASE and subqueries

In [ ]:
show('''
SELECT name,
       CASE
         WHEN salary IS NULL THEN 'Unknown'
         WHEN salary >= 90000 THEN 'High'
         WHEN salary >= 70000 THEN 'Mid'
         ELSE 'Entry'
       END AS salary_band
FROM employees
ORDER BY employee_id;
''')

In [ ]:
# Employees earning above the overall average.
show('''
SELECT name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC;
''')

## 6. CTEs and window functions

In [ ]:
# DENSE_RANK assigns equal ranks to ties without gaps.
show('''
SELECT name, department_id, salary,
       DENSE_RANK() OVER (
         PARTITION BY department_id
         ORDER BY salary DESC
       ) AS salary_rank
FROM employees
WHERE salary IS NOT NULL
ORDER BY department_id, salary_rank, name;
''')

In [ ]:
# Running order amount, with deterministic ordering.
show('''
SELECT order_id, order_date, amount,
       SUM(amount) OVER (
         ORDER BY order_date, order_id
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_amount
FROM orders
ORDER BY order_date, order_id;
''')

In [ ]:
# Compare each order date's revenue with the previous date present in the data.
show('''
WITH daily AS (
  SELECT order_date, SUM(amount) AS revenue
  FROM orders
  GROUP BY order_date
)
SELECT order_date, revenue,
       LAG(revenue) OVER (ORDER BY order_date) AS previous_revenue,
       revenue - LAG(revenue) OVER (ORDER BY order_date) AS change_from_previous
FROM daily
ORDER BY order_date;
''')

## 7. Find the second-highest distinct salary

In [ ]:
show('''
SELECT MAX(salary) AS second_highest_distinct_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
''')

## 8. Latest order per customer

In [ ]:
show('''
WITH ranked_orders AS (
  SELECT order_id, customer_id, order_date, amount,
         ROW_NUMBER() OVER (
           PARTITION BY customer_id
           ORDER BY order_date DESC, order_id DESC
         ) AS rn
  FROM orders
)
SELECT order_id, customer_id, order_date, amount
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_id;
''')

## 9. Query plan and indexes

SQLite can show a high-level query plan with `EXPLAIN QUERY PLAN`. Indexes can improve some reads but add storage and write-maintenance costs.

In [ ]:
conn.execute("CREATE INDEX idx_employees_department_salary ON employees(department_id, salary)")
pd.DataFrame(
    conn.execute(
        "EXPLAIN QUERY PLAN SELECT name FROM employees WHERE department_id = 20 AND salary >= 70000"
    ).fetchall(),
    columns=["id", "parent", "notused", "detail"]
)

## 10. Mini challenges

Try these without looking at `solution.md`:

1. Find departments with no employees.
2. Return all employees tied for the highest salary in each department.
3. Count orders and total revenue by category.
4. Find customers who have no orders (you will need to create a `customers` table first).
5. Rewrite one query as a CTE and explain why the CTE helps readability.

Close the connection when finished if desired: `conn.close()`.